# Appendix A · The Excel ↔ Python Bridge
### Financial Analytics Course

Finance runs on Excel. You now run on Python. This notebook is the peace treaty — the four skills that let both be true:

1. **Read** real-world Excel files (the messy kind) into pandas
2. **Write** boss-ready, formatted Excel reports from Python
3. Know **which tool wins which job** (the honest table)
4. The two **workflow patterns** every finance team eventually converges on

*Requires: `pip install openpyxl` (pandas' Excel engine).*

In [ ]:
import pandas as pd
import numpy as np

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
cli = pd.read_csv(BASE + "client_book.csv")

# First: manufacture a realistically annoying Excel file to practise on -
# two sheets, junk title rows, the kind of thing a colleague actually sends you.
with pd.ExcelWriter("colleague_file.xlsx", engine="openpyxl") as xw:
    summary = cli.groupby("segment").agg(clients=("client_id","count"), aum=("aum_inr","sum"))
    summary.to_excel(xw, sheet_name="Summary", startrow=3)          # data starts row 4!
    cli.head(200).to_excel(xw, sheet_name="Raw Clients", index=False)
print("colleague_file.xlsx created (with a 3-row junk header on Summary, as tradition demands)")

---
## 1. Reading the real world

`pd.read_excel` is `read_csv`'s sibling — with three arguments that handle 90% of real files:

In [ ]:
# sheet_name: which tab. skiprows: the junk title rows. usecols: ignore the decoration columns.
summary = pd.read_excel("colleague_file.xlsx", sheet_name="Summary", skiprows=3)
raw     = pd.read_excel("colleague_file.xlsx", sheet_name="Raw Clients")
everything = pd.read_excel("colleague_file.xlsx", sheet_name=None)   # dict of ALL sheets

print(summary)
print("\nSheets found:", list(everything.keys()))
print("\nAfter this line, it's a DataFrame - Module 3 takes over. Run the first-look ritual as always:")
print("Excel files LIE more than CSVs (merged cells, numbers-stored-as-text, hidden rows).")
raw.info()

**The Excel-specific traps to expect** (add these to your Module 3 ritual whenever the source is .xlsx): numbers stored as text (`info()` shows `object` where you expected numbers → `pd.to_numeric(col, errors="coerce")`); dates as Excel serial numbers (45123 instead of a date → `pd.to_datetime(col, unit="D", origin="1899-12-30")`); merged cells arriving as one value + NaNs (→ `.ffill()` *only* where the merge semantics justify it); and phantom columns of pure NaN from stray formatting (→ `dropna(axis=1, how="all")`).

---
## 2. Writing a report your boss opens and smiles at

`df.to_excel()` alone produces a raw dump — technically Excel, spiritually a CSV. A *report* needs number formats, widths, and a title block. openpyxl (which pandas uses underneath) does all of it:

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# The analysis (Module 3 work, one line each)
report = cli.groupby("segment").agg(
    clients=("client_id","count"),
    total_aum=("aum_inr","sum"),
    avg_aum=("aum_inr","mean"),
    churn_rate=("churned","mean"),
).reset_index()

with pd.ExcelWriter("segment_report.xlsx", engine="openpyxl") as xw:
    report.to_excel(xw, sheet_name="Segment Report", index=False, startrow=2)
    ws = xw.sheets["Segment Report"]

    # Title block
    ws["A1"] = "Wealth Book - Segment Report"
    ws["A1"].font = Font(size=14, bold=True, color="1F4E79")
    ws["A2"] = f"Generated from client_book.csv | rows: {len(cli):,} | course synthetic data"
    ws["A2"].font = Font(size=9, italic=True, color="808080")

    # Header row styling
    for cell in ws[3]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="2563EB")
        cell.alignment = Alignment(horizontal="center")

    # Number formats: THE difference between a dump and a report
    for row in range(4, 4+len(report)):
        ws[f"C{row}"].number_format = '#,##0'        # total AUM: thousands separators
        ws[f"D{row}"].number_format = '#,##0'        # avg AUM
        ws[f"E{row}"].number_format = '0.0%'         # churn as a percentage

    # Column widths
    for i, col in enumerate(report.columns, start=1):
        ws.column_dimensions[get_column_letter(i)].width = max(14, len(str(col))+4)

print("segment_report.xlsx written - formatted, titled, boss-ready.")
print(report.round(3).to_string(index=False))

Thirty lines, and they're the *same* thirty lines for every report you'll ever ship — wrap them in a function once (`write_report(df, path, title)`) and formatted Excel becomes a one-liner forever. **This is the appendix's core promise:** the monthly pack that took an afternoon of manual formatting becomes `python make_report.py`, identical every month, with lineage (Module 1) for free because the code IS the documentation.

*(Going further: charts-in-Excel via openpyxl exist but get painful fast — the pragmatic pattern is numbers in Excel, charts as embedded images or a linked PDF. And `xlsxwriter` is an alternative engine with nicer chart support; openpyxl's advantage is it also READS.)*

---
## 3. The honest division of labour

| The job | Winner | Why |
|---|---|---|
| Quick what-if a colleague can poke | **Excel** | Immediacy; everyone can edit; no environment |
| Anything repeated monthly | **Python** | One keypress, identical every time, auditable |
| 500+ thousand rows | **Python** | Excel slows, then silently truncates old formats |
| The deliverable an exec opens | **Excel** (written BY Python) | Their tool, your pipeline - this notebook's pattern |
| Collaborative input collection | **Excel/Sheets** | Humans type into grids; let them |
| Anything needing lineage/audit | **Python** | Cell F47 in tab 23 is where lineage goes to die |
| Statistics, simulation, optimisation | **Python** | Modules 5-12 don't fit in cells |

The professional stance is neither snobbery nor surrender: **Excel is finance's *presentation and collection* layer; Python is its *computation* layer.** Fighting that reality wastes careers; bridging it (this notebook) compounds them.

## 4. The two workflow patterns

**Pattern 1 - Python computes → Excel presents** (what you just built): pipeline in Python, formatted .xlsx out, humans consume. The monthly-pack pattern.

**Pattern 2 - Excel collects → Python processes**: budget templates, RM input sheets, ops trackers — humans fill grids, `read_excel` ingests them (with validation! humans type '10 Cr' into number columns), Python consolidates. The planning-season pattern.

Most finance teams run both simultaneously. The analyst who can build either bridge is the one who ends up owning the process.

### ✏️ Exercises
1. **The report function:** wrap section 2 into `write_report(df, path, title, pct_cols=[], num_cols=[])` and ship the city-level version of the same report with one call.
2. **The validation gate:** add a deliberately bad row to a copy of the colleague file (text in the AUM column). Write the read-side check that catches it and reports the offending row *before* any analysis runs — Pattern 2's survival skill.
3. **Round-trip fidelity:** write `report` to Excel, read it back, and `assert` the churn numbers survived unchanged. (They will — but now you have the habit for the day a format string silently rounds something that mattered.)

*AI disclosure: ______*

In [ ]:
# workspace
